# 01 Update US and Singapore Data

Restart the kernel and select Run All. Install `yfinance` first if needed.

Download nine US ETFs, ES3.SI, A35.SI, and SGD=X from 2011-10-01 with `auto_adjust=False`, then overwrite the two price files:

| File | Contents | Consumers |
|---|---|---|
| `data/raw/etf_daily.csv` | All tickers in the original yfinance multi-level header format | 02, 05, 06, 07, 08 |
| `data/raw/sti_daily.csv` | One row per ES3 trading day; ES3, A35 (`bond_` prefix), and `usdsgd` | 03, 07, 08 |

Before replacement, back up existing files as `etf_daily_backup.csv` and `sti_daily_backup.csv`, replacing the preceding backup. Discard unfinished current-day rows before 16:30 New York time for US data and 17:30 Singapore time for Singapore data. If any US ETF download fails, neither file is overwritten. A Singapore download failure can skip the Singapore output only.


In [ ]:
from pathlib import Path
import os
import json

def find_pipeline_dir():
    """Supports execution from the project root, pipeline directory and project subdirectories."""
    roots = ([Path(os.environ['ETF_PROJECT_ROOT']).expanduser()] if os.environ.get('ETF_PROJECT_ROOT') else [])
    roots += [Path.cwd(), *Path.cwd().parents]
    for root in roots:
        for candidate in (root / 'pipeline', root):
            if (candidate / '00_engine.ipynb').is_file():
                return candidate.resolve()
    raise FileNotFoundError('Cannot find pipeline Directory.Please set PyCharm Set the working directory to the project root, or set ETF_PROJECT_ROOT.')

PIPELINE_DIR = find_pipeline_dir()

def load_notebook_definitions(filename):
    """Load functions into an independent namespace; Do not execute the loaded notebook entry cell."""
    path = next(iter(sorted(PIPELINE_DIR.rglob(filename))), PIPELINE_DIR / filename)   # At pipeline Search subdirectories
    notebook = json.loads(path.read_text(encoding='utf-8'))
    namespace = {'__name__': 'pipeline_module'}
    for number, item in enumerate(notebook['cells']):
        if item['cell_type'] == 'code' and 'pipeline-entry' not in item.get('metadata', {}).get('tags', []):
            exec(compile(''.join(item['source']), f'{path}:cell{number}', 'exec'), namespace)
    return namespace

_engine = load_notebook_definitions('00_engine.ipynb')
for _name, _value in _engine.items():
    if not _name.startswith('_'):
        globals()[_name] = _value


In [ ]:
import shutil

DATA_DIR = PIPELINE_DIR.parent / 'data' / 'raw'
ETF_CSV, STI_CSV = DATA_DIR / 'etf_daily.csv', DATA_DIR / 'sti_daily.csv'
SG = {'ES3': 'ES3.SI', 'A35': 'A35.SI', 'FX': 'SGD=X'}


def drop_unfinished(df, tz, close_minutes, label):
    """Before the local market close, "Today"This row contains intraday prices, Discard."""
    now = pd.Timestamp.now(tz=tz)
    if len(df) and df.index[-1].date() >= now.date() and now.hour * 60 + now.minute < close_minutes:
        print(f'{label}: Discard the current days unfinished data {df.index[-1].date()} (Local time {now:%H:%M} Market has not closed)')
        df = df.iloc[:-1]
    return df


def has(data, t):
    return ('Adj Close', t) in data and not data['Adj Close'][t].dropna().empty


def download_all(start='2011-10-01'):
    import yfinance as yf
    data = yf.download(TICKS + list(SG.values()), start=start, auto_adjust=False)
    if data.empty:
        raise ValueError('Download returned no data, No files were overwritten.')
    missing = [t for t in TICKS if not has(data, t)]
    if missing:
        raise ValueError(f'Download is missing US tickers: {missing}; No files were overwritten.')
    return data


def build_etf_daily(data):
    return drop_unfinished(data, 'America/New_York', 16 * 60 + 30, 'etf_daily (US equities)')


def build_sti_daily(data):
    """and 03 notebook load_sti() Matches the expected input format: One row per ES3 Trading_days."""
    def ohlc(t, prefix=''):
        return pd.DataFrame({f'{prefix}open': data['Open'][t], f'{prefix}high': data['High'][t],
                             f'{prefix}low': data['Low'][t], f'{prefix}close': data['Close'][t],
                             f'{prefix}adj_close': data['Adj Close'][t], f'{prefix}volume': data['Volume'][t]})
    out = ohlc(SG['ES3']).dropna(subset=['close'])
    out = out.join(ohlc(SG['A35'], 'bond_').reindex(out.index))     # A35 Leave missing days blank, 03 Handled during loading by the notebook"Unchanged price"Handle
    fx = data['Close'][SG['FX']].dropna()
    out['usdsgd'] = fx.reindex(out.index.union(fx.index)).ffill().reindex(out.index)   # Use the latest exchange rate available on or before that date
    out.index.name = 'Date'
    return drop_unfinished(out, 'Asia/Singapore', 17 * 60 + 30, 'sti_daily (Singapore)')


def overwrite(df, target):
    if target.is_file():
        shutil.copy2(target, target.with_name(target.stem + '_backup.csv'))
    df.to_csv(target)


def main(download=True):
    if not download:
        print('Not downloaded (DOWNLOAD = False).')
        return None
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    data = download_all()

    etf = build_etf_daily(data)
    overwrite(etf, ETF_CSV)
    last_us = etf['Adj Close'][TICKS].dropna().index.max().date()
    print(f'Overwritten {ETF_CSV.name} (Old file backed up as {ETF_CSV.stem}_backup.csv)| Latest complete US trading day: {last_us}')

    sti = None
    lost = [t for t in SG.values() if not has(data, t)]
    if lost:
        print(f'⚠ Singapore tickers {lost} Not downloaded, {STI_CSV.name} Keep unchanged.')
    else:
        sti = build_sti_daily(data)
        overwrite(sti, STI_CSV)
        print(f'Overwritten {STI_CSV.name} (Old file backed up as {STI_CSV.stem}_backup.csv)| '
              f'{sti.index[0].date()} ~ {sti.index[-1].date()} | A35 Days with data: {int(sti["bond_close"].notna().sum())}')
    return {'etf_daily': etf, 'sti_daily': sti}


In [ ]:
DOWNLOAD = True      # Change to False then do not download
results = main(download=DOWNLOAD)
